### Performing EDA on OLIST Dataset

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as snb

In [4]:
customer_data = pd.read_csv("..\data\olist_customers_dataset.csv")
olist_orders = pd.read_csv("..\data\olist_orders_dataset.csv")
olist_order_items = pd.read_csv("..\data\olist_order_items_dataset.csv")
olist_order_payments = pd.read_csv("..\data\olist_order_payments_dataset.csv")
olist_order_reviews = pd.read_csv("..\data\olist_order_reviews_dataset.csv")
olist_products = pd.read_csv("..\data\olist_products_dataset.csv")
olist_product_category = pd.read_csv("..\data\product_category_name_translation.csv")

In [31]:
customer_data.head()
olist_orders.head()
olist_order_items.head()
# olist_order_payments.head()
# # olist_order_reviews.head()
olist_products.head()
# olist_product_category.head()


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


### Checking Multiple Payments issue

In [6]:

counts = olist_order_payments["order_id"].value_counts()
duplicate_ids = counts[counts>1].index
olist_order_payments[olist_order_payments["order_id"].isin(duplicate_ids)].sort_values(by="order_id")

#Faster Way
olist_order_payments[olist_order_payments.duplicated(subset=['order_id'], keep=False)].sort_values(by='order_id')

,order_id,payment_sequential,payment_type,payment_installments,payment_value
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
...,...,...,...,...,...
21648,ffa1dd97810de91a03abd7bd76d2fed1,2,voucher,1,418.73
32912,ffa39020fe7c8a3e907320e1bec4b985,1,credit_card,1,7.13
3009,ffa39020fe7c8a3e907320e1bec4b985,2,voucher,1,64.01
75188,ffc730a0615d28ec19f9cad02cb41442,1,credit_card,1,14.76


In [7]:
# Grouping multiple orders in payments
unique_order_payments = olist_order_payments.groupby(["order_id"]).agg(max_installments=('payment_installments', 'max'),
                                                                       max_methods = ('payment_sequential', 'max'),
                                                                       total_payment = ('payment_value','sum')).reset_index()
primary_payments=olist_order_payments.sort_values(by="payment_value", ascending=False).drop_duplicates(subset="order_id")
primary_payments

# Final Cleaned Payments Data
olist_cleaned_payments = pd.merge(primary_payments[['order_id','payment_type']], unique_order_payments,
                                  on="order_id", how="inner")
olist_cleaned_payments.head(26)

,order_id,payment_type,max_installments,max_methods,total_payment
0,03caa2c082116e1d31e67e9ae3700499,credit_card,1,1,13664.08
1,736e1922ae60d0d6a89247b851902527,boleto,1,1,7274.88
2,0812eb902a67711a1cb742b3cdaa65ae,credit_card,8,1,6929.31
3,fefacc66af859508bf1a7934eab1e97f,boleto,1,1,6922.21
4,f5136e38d1a14a4dbd87dff67da82701,boleto,1,1,6726.66
5,2cc9089445046817a7539d90805e6e5a,boleto,1,1,6081.54
6,a96610ab360d42a2e5335a3998b4718a,credit_card,10,1,4950.34
7,b4c4b76c642808cbe472a32b86cddc95,credit_card,5,1,4809.44
8,199af31afc78c699f0dbf71fb178d4d4,credit_card,8,1,4764.34
9,8dbc85d1447242f3b127dda390d56e19,credit_card,8,1,4681.78


### Merging customer and order data

In [8]:

customer_order_data = pd.merge(customer_data, olist_orders, how="left", on="customer_id")
customer_order_data.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05 00:00:00
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06 00:00:00
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13 00:00:00
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10 00:00:00
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15 00:00:00


#### pd.merge ignores Index!!

In [9]:
# Merging Customer and Payment data
master_customer_data = pd.merge(customer_order_data, olist_cleaned_payments, on="order_id", how="inner")
master_customer_data.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_type,max_installments,max_methods,total_payment
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05 00:00:00,credit_card,2,1,146.87
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06 00:00:00,credit_card,8,1,335.48
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13 00:00:00,credit_card,7,1,157.73
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10 00:00:00,credit_card,1,1,173.30
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15 00:00:00,credit_card,8,1,252.25


### order_items and product data handling

.nunique(), .duplicated(), .isin() return series. To conver to dataframe -> .reset_index()

.isnull()

In [ ]:
olist_order_items[olist_order_items.duplicated(subset=["order_id"],keep=False)].head(10)
olist_order_items[olist_order_items["order_item_id"]>20]
olist_order_items[olist_order_items["order_id"]=="ffb9a9cd00c74c11c24aa30b3d78e03b"]
# There are order_ids which have Different products!!!

mixed_product_counts = olist_order_items.groupby('order_id')['product_id'].nunique()  
mixed_product_order_ids = mixed_product_counts[mixed_product_counts>1].index
mixed_product_orders = olist_order_items[olist_order_items['order_id'].isin(mixed_product_order_ids)]
mixed_product_orders.head(25)
mixed_product_orders.sort_values(by=['order_id', 'order_item_id']).head(6)
# All different products in an order!!

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
80,002f98c0f7efd42638ed6100ca699b42,1,d41dc2f2979f52d75d78714b378d4068,7299e27ed73d2ad986de7f7c77d919fa,2017-08-10 09:30:15,8.99,32.57
81,002f98c0f7efd42638ed6100ca699b42,2,880be32f4db1d9f6e2bec38fb6ac23ab,fa40cc5b934574b62717c68f3d678b6d,2017-08-10 09:30:15,44.90,7.16
91,00337fe25a3780b3424d9ad7c5a4b35e,1,1f9799a175f50c9fa725984775cac5c5,cfb1a033743668a192316f3c6d1d2671,2017-09-29 17:50:16,59.90,9.94
92,00337fe25a3780b3424d9ad7c5a4b35e,2,13944d17b257432717fd260e69853140,cfb1a033743668a192316f3c6d1d2671,2017-09-29 17:50:16,59.90,9.94
151,005d9a5423d47281ac463a968b3936fb,1,fb7a100ec8c7b34f60cec22b1a9a10e0,d98eec89afa3380e14463da2aabaea72,2017-10-24 12:28:16,49.99,18.12
152,005d9a5423d47281ac463a968b3936fb,2,4c3ae5db49258df0784827bdacf3b396,d98eec89afa3380e14463da2aabaea72,2017-10-24 12:28:16,24.99,13.58


#### Merging related product tables: Order_items, products, product_category_name

Checking nulls -> 1.isna().sum(), 2. isna().any(), isna.any(axis=1) checks horizontally

Filling nulls -> .fillna(df_column) only corresponding row/index of df_column will be filled!!

In [91]:
olist_detailed_order = pd.merge(olist_order_items, olist_products, how="left", on="product_id")
olist_product_category.head()
olist_detailed_order = pd.merge(olist_detailed_order, olist_product_category, how="left", on="product_category_name")
olist_detailed_order = olist_detailed_order.drop(["product_name_lenght", "product_description_lenght",
                                                  "product_photos_qty","product_weight_g",
                                                  "product_length_cm","product_width_cm","product_height_cm"],axis=1)
olist_detailed_order[olist_detailed_order['order_id']=="ffb9a9cd00c74c11c24aa30b3d78e03b"]

#Checking null values
olist_detailed_order.isna().sum(axis=0)
# Filling null valuees

olist_detailed_order['product_category_name_english'] = olist_detailed_order['product_category_name_english'].fillna(
                                                                            olist_detailed_order['product_category_name']).fillna(
                                                                                'Unknown'
                                                                            )
                     
olist_detailed_order = olist_detailed_order.drop(["product_category_name"], axis=1)
olist_detailed_order = olist_detailed_order.rename(columns={"product_category_name_english":"product_category_name"})
olist_detailed_order.isnull().sum()
olist_detailed_order.head()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,garden_tools
